In [10]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.transformer_lstm import Transformer_LSTM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_CLS  = Transformer_LSTM
MODEL_NAME = MODEL_CLS.__name__


device: NVIDIA A30


In [11]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv("../CSV_Files/val.csv").to_numpy()
    test  = pd.read_csv("../CSV_Files/test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh Transformer-LSTM with Xavier init on Linear layers.
    LSTM and MultiheadAttention use their own PyTorch defaults."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-3, weight_decay=1e-4,
                 seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(Transformer_LSTM, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  shapes: train={tuple(X_tr.shape)} val={tuple(X_va.shape)} "
              f"test={tuple(X_te.shape)}")
        print(f"  train labels: {np.bincount(y_tr.numpy())}")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # training
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # inference timing — backbone forward + argmax
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # test evaluation
    y_true, y_pred = predict_all(model, test_loader, device)

    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":  scenario_idx,
        "model":     MODEL_NAME,
        "n_train":   len(X_tr),
        "accuracy":  acc,
        "precision": p,
        "recall":    r,
        "f1":        f,
        "confusion": cm,
        "n_params":  n_params,
        "train_sec": round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


def run_scenario_multi_seed(scenario_idx, scenario_dir, device,
                            epochs=200, lr=1e-3, seeds=range(10),
                            verbose=True):
    """Run a scenario across multiple seeds, return mean ± std."""
    seed_runs = []
    for s in seeds:
        if verbose:
            print(f"\n--- Scenario {scenario_idx}, seed={s} ---")
        r = run_scenario(scenario_idx, scenario_dir, device,
                         epochs=epochs, lr=lr, seed=s,
                         verbose=False)
        seed_runs.append(r)
        if verbose:
            print(f"  seed {s}: acc={r['accuracy']:.4f}  F1={r['f1']:.4f}")

    metric_keys = ["accuracy", "precision", "recall", "f1"]
    agg = {
        "scenario":  scenario_idx,
        "model":     seed_runs[0]["model"],
        "n_train":   seed_runs[0]["n_train"],
        "n_params":  seed_runs[0]["n_params"],
        "n_seeds":   len(list(seeds)),
    }
    for k in metric_keys:
        vals = np.array([r[k] for r in seed_runs])
        agg[k] = vals.mean()
        agg[f"{k}_std"] = vals.std(ddof=1)

    agg["train_sec"] = np.mean([r["train_sec"] for r in seed_runs])
    agg["inf_ms_per_sample"] = np.mean([r["inf_ms_per_sample"] for r in seed_runs])
    agg["peak_mem_mb"] = np.mean([r["peak_mem_mb"] for r in seed_runs])

    # Keep individual seed results for variance analysis
    agg["per_seed_f1"] = [r["f1"] for r in seed_runs]
    agg["per_seed_acc"] = [r["accuracy"] for r in seed_runs]

    if verbose:
        print(f"\n  === Scenario {scenario_idx} aggregated over {len(list(seeds))} seeds ===")
        print(f"  F1: {agg['f1']:.4f} ± {agg['f1_std']:.4f}")
        print(f"  Acc: {agg['accuracy']:.4f} ± {agg['accuracy_std']:.4f}")

    return agg

In [12]:
SEEDS = [i for i in range(10)]   # 5 seeds — Li used 10, but 5 will catch most variance

results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario_multi_seed(i, sc_dir, device,
                                epochs=200, lr=1e-3, seeds=SEEDS)
    results.append(r)


--- Scenario 1, seed=0 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 0: acc=0.5581  F1=0.5523

--- Scenario 1, seed=1 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 1: acc=0.6181  F1=0.6098

--- Scenario 1, seed=2 ---


...

  seed 7: acc=0.2156  F1=0.1474

--- Scenario 6, seed=8 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 8: acc=0.1600  F1=0.0859

--- Scenario 6, seed=9 ---


/home/maddie/miniconda3/envs/ad/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


  seed 9: acc=0.1406  F1=0.0680

  === Scenario 6 aggregated over 10 seeds ===
  F1: 0.1075 ± 0.0241
  Acc: 0.1752 ± 0.0279


In [13]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "accuracy_std",
                       "precision", "precision_std",
                       "recall", "recall_std",
                       "f1", "f1_std",
                       "n_params", "train_sec",
                       "inf_ms_per_sample", "peak_mem_mb")}
    for r in results
])

cols_round = ["accuracy", "accuracy_std",
              "precision", "precision_std",
              "recall", "recall_std",
              "f1", "f1_std"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios (mean over {len(SEEDS)} seeds) ===")
print(summary.to_string(index=False))
summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)

# Also save per-seed details
per_seed = pd.DataFrame([
    {"scenario": r["scenario"], "seed": s, "f1": f1, "acc": acc}
    for r in results
    for s, f1, acc in zip(SEEDS, r["per_seed_f1"], r["per_seed_acc"])
])
per_seed.to_csv(f"{MODEL_NAME.lower()}_per_seed.csv", index=False)


=== Transformer_LSTM summary across scenarios (mean over 10 seeds) ===
 scenario            model  n_train  accuracy  accuracy_std  precision  precision_std  recall  recall_std     f1  f1_std  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 Transformer_LSTM     7860    0.7286        0.0855     0.7543         0.0651  0.7286      0.0855 0.7115  0.0812      5397    338.208            0.00197         26.7
        2 Transformer_LSTM     3912    0.6710        0.1130     0.7165         0.0956  0.6710      0.1130 0.6525  0.1121      5397    172.967            0.00220         26.7
        3 Transformer_LSTM      778    0.4616        0.0743     0.4983         0.1392  0.4616      0.0743 0.4172  0.0897      5397     48.073            0.00220         26.7
        4 Transformer_LSTM      394    0.3461        0.0657     0.3428         0.1354  0.3461      0.0657 0.2819  0.0781      5397     27.776            0.00213         26.7
        5 Transformer_LSTM      236    0.3062        0.044

In [14]:
# for r in results:
#     print(f"\nScenario {r['scenario']}  ({r['model']}, "
#           f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
#     print(r["confusion"])
